In [1]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "shared")
root = Path.cwd().resolve()
while root != root.parent and not (root / "shared").exists():
    root = root.parent

if not (root / "shared").exists():
    raise RuntimeError("Could not find project root containing 'shared/'")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Project root:", root)
print("sys.path[0]:", sys.path[0])

from shared.paths import get_experiment_paths
from shared.seed import seed_everything
from shared.visualization import show_grid, save_grid

Project root: D:\Projects\GANs
sys.path[0]: D:\Projects\GANs


In [2]:
EXP = "02_dcgan"
paths = get_experiment_paths(EXP)

seed_everything(42)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_ds = datasets.MNIST(
    root=str(paths.datasets_dir / "MNIST"),
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from shared.weight_init import dcgan_weights_init

In [5]:
z_dim = 100
feat = 64

class DCGAN_Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=1, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, feat*4, 7, 1, 0, bias=False),  # 1x1 -> 7x7
            nn.BatchNorm2d(feat*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*4, feat*2, 4, 2, 1, bias=False), # 7x7 -> 14x14
            nn.BatchNorm2d(feat*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*2, feat, 4, 2, 1, bias=False),   # 14x14 -> 28x28
            nn.BatchNorm2d(feat),
            nn.ReLU(True),

            nn.Conv2d(feat, img_channels, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

class DCGAN_Discriminator(nn.Module):
    def __init__(self, img_channels=1, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, feat, 4, 2, 1),                 # 28 -> 14
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat, feat*2, 4, 2, 1, bias=False),           # 14 -> 7
            nn.BatchNorm2d(feat*2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*2, feat*4, 3, 2, 1, bias=False),         # 7 -> 4
            nn.BatchNorm2d(feat*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*4, 1, 4, 1, 0),                          # 4 -> 1
        )

    def forward(self, x):
        out = self.net(x)    # (N,1,1,1)
        return out.view(-1)  # (N,)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

G = DCGAN_Generator(z_dim=z_dim, feat=feat).to(device)
D = DCGAN_Discriminator(feat=feat).to(device)

G.apply(dcgan_weights_init)
D.apply(dcgan_weights_init)

g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

epochs = 20
fixed_z = torch.randn(64, z_dim, 1, 1, device=device)

for epoch in range(1, epochs + 1):
    G.train(); D.train()

    for real, _ in train_loader:
        real = real.to(device, non_blocking=True)
        b = real.size(0)

        # Train D
        z = torch.randn(b, z_dim, 1, 1, device=device)
        fake = G(z).detach()

        d_real = D(real)
        d_fake = D(fake)

        real_labels = torch.full((b,), 0.9, device=device)  # label smoothing
        fake_labels = torch.zeros(b, device=device)

        d_loss = (criterion(d_real, real_labels) + criterion(d_fake, fake_labels)) / 2

        d_opt.zero_grad(set_to_none=True)
        d_loss.backward()
        d_opt.step()

        # Train G
        z = torch.randn(b, z_dim, 1, 1, device=device)
        fake = G(z)
        d_fake_for_g = D(fake)

        g_loss = criterion(d_fake_for_g, torch.ones(b, device=device))

        g_opt.zero_grad(set_to_none=True)
        g_loss.backward()
        g_opt.step()

    G.eval()
    with torch.no_grad():
        samples = G(fixed_z)

    save_grid(samples, paths.samples_dir / f"epoch_{epoch:03d}.png", nrow=8)

    torch.save({
        "epoch": epoch,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "g_opt": g_opt.state_dict(),
        "d_opt": d_opt.state_dict(),
        "z_dim": z_dim,
        "feat": feat
    }, paths.models_dir / f"ckpt_epoch_{epoch:03d}.pth")

    print(f"[DCGAN] Epoch {epoch}/{epochs} | d_loss={d_loss.item():.4f} | g_loss={g_loss.item():.4f}")

[DCGAN] Epoch 1/20 | d_loss=0.4648 | g_loss=0.9392
[DCGAN] Epoch 2/20 | d_loss=0.5069 | g_loss=1.3197
[DCGAN] Epoch 3/20 | d_loss=0.5162 | g_loss=1.0793
[DCGAN] Epoch 4/20 | d_loss=0.5236 | g_loss=1.3182
[DCGAN] Epoch 5/20 | d_loss=0.4764 | g_loss=1.5766
[DCGAN] Epoch 6/20 | d_loss=0.4512 | g_loss=1.8199
[DCGAN] Epoch 7/20 | d_loss=0.4732 | g_loss=1.7017
[DCGAN] Epoch 8/20 | d_loss=0.5649 | g_loss=2.7638
[DCGAN] Epoch 9/20 | d_loss=0.5581 | g_loss=1.6749
[DCGAN] Epoch 10/20 | d_loss=0.3706 | g_loss=1.4926
[DCGAN] Epoch 11/20 | d_loss=0.4514 | g_loss=1.8439
[DCGAN] Epoch 12/20 | d_loss=0.4389 | g_loss=1.8793
[DCGAN] Epoch 13/20 | d_loss=0.3822 | g_loss=1.6763
[DCGAN] Epoch 14/20 | d_loss=0.4230 | g_loss=1.6557
[DCGAN] Epoch 15/20 | d_loss=0.4761 | g_loss=2.3384
[DCGAN] Epoch 16/20 | d_loss=0.4100 | g_loss=2.1349
[DCGAN] Epoch 17/20 | d_loss=0.3363 | g_loss=1.8413
[DCGAN] Epoch 18/20 | d_loss=0.3813 | g_loss=1.6034
[DCGAN] Epoch 19/20 | d_loss=0.4621 | g_loss=1.6900
[DCGAN] Epoch 20/20 |